In [ ]:
import os, sys, subprocess, platform
print(platform.python_version())
subprocess.run(['nvidia-smi'], check=False)
subprocess.run(['bash','-lc','nproc; free -h'], check=False)
import torch
print('torch', torch.__version__, 'CUDA', torch.version.cuda, 'available', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU runtime')

In [ ]:
# Select exactly one run. Keep RUN_MODE and its run_id unchanged when resuming.
RUN_MODE = 'warm'  # use 'scratch' for the independent random-initialization control
DRIVE_ROOT = '/content/drive/MyDrive/connect4'
MAX_SEGMENT_MINUTES = 30
PHASE_C_GPU_MINUTES = 120
PER_RUN_GPU_MINUTES = 60
DATASET_NAME = 'solverlabels_20261121_seed20261121_v3_30k'
SOURCE_ZIP = f'{DRIVE_ROOT}/source_bundle_20261121_v12.zip'
SOURCE_MANIFEST = f'{DRIVE_ROOT}/source_bundle_20261121_v12.manifest.json'
BEST_CNN_CHECKPOINT = f'{DRIVE_ROOT}/checkpoints/cnn_v3_30k_best.pt'  # validation-selected after Phase B
if RUN_MODE == 'warm':
    RUN_ID, SEED, INIT_CHECKPOINT = 'az_v2_warm_seed20261121', 20261121, BEST_CNN_CHECKPOINT
elif RUN_MODE == 'scratch':
    RUN_ID, SEED, INIT_CHECKPOINT = 'az_v2_scratch_seed20261122', 20261122, None
else:
    raise ValueError("RUN_MODE must be 'warm' or 'scratch'")
MAX_ITERS = 40
GAMES_PER_ITERATION = 32
PARALLEL_GAMES = 16
SIMULATIONS = 400
TRAINING_STEPS = 200
BATCH_SIZE = 128
SNAPSHOT_INTERVAL = 5
SMOKE = False  # smoke runs are not research artifacts


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
root = Path(DRIVE_ROOT); root.mkdir(parents=True, exist_ok=True)
dataset = root/'data'/DATASET_NAME
assert Path(SOURCE_ZIP).is_file(), 'Upload source.zip first'
assert (dataset/'manifest.json').is_file(), 'Upload the complete dataset package first'
if RUN_MODE == 'warm':
    assert Path(INIT_CHECKPOINT).is_file(), 'Upload the validation-selected CNN v3 checkpoint first.'


In [ ]:
import zipfile, shutil
repo = Path('/content/connect4-project')
if repo.exists(): shutil.rmtree(repo)
repo.mkdir()
import hashlib, json
bundle = json.loads(Path(SOURCE_MANIFEST).read_text())
assert hashlib.sha256(Path(SOURCE_ZIP).read_bytes()).hexdigest() == bundle['zip_sha256'], 'source ZIP checksum mismatch'
with zipfile.ZipFile(SOURCE_ZIP) as z: z.extractall(repo)
candidates = [p for p in repo.iterdir() if p.is_dir()]
if not (repo/'connect4').is_dir() and len(candidates)==1: repo = candidates[0]
def tree_hash(root):
    h=hashlib.sha256()
    for p in sorted(root.rglob('*.py')):
        if any(part in {'.git','.venv','.venv-gpu','__pycache__','.pytest_cache'} for part in p.parts): continue
        h.update(p.relative_to(root).as_posix().encode('utf-8')); h.update(p.read_bytes())
    return h.hexdigest()
assert tree_hash(repo) == bundle['source_tree_sha256'], 'source tree differs from packaged snapshot'


In [ ]:
%cd $repo
!pip install -r requirements-colab.txt

In [ ]:
import hashlib, json
data_manifest = json.loads((dataset/'manifest.json').read_text())
for name, expected in data_manifest['files_sha256'].items():
    actual = hashlib.sha256((dataset/name).read_bytes()).hexdigest()
    assert actual == expected, f'dataset checksum mismatch: {name}'
print('frozen test SHA256:', data_manifest['frozen_test_sha256'])

In [ ]:
import os, json, time, shutil
os.environ['PYTHONPATH'] = str(repo)
out = root/'artifacts'; out.mkdir(parents=True, exist_ok=True); run_dir = out/RUN_ID
budget_path = out/'az_v2_phase_c_budget.json'
budget = json.loads(budget_path.read_text()) if budget_path.exists() else {'phase_limit_seconds': PHASE_C_GPU_MINUTES*60, 'per_run_limit_seconds': PER_RUN_GPU_MINUTES*60, 'runs': {}}
assert budget['phase_limit_seconds'] == PHASE_C_GPU_MINUTES*60 and budget['per_run_limit_seconds'] == PER_RUN_GPU_MINUTES*60, 'Budget tracker limits differ; do not reset or edit it.'
assert Path(INIT_CHECKPOINT).is_file() if INIT_CHECKPOINT else True, 'Warm-start checkpoint missing.'
run_used = float(budget['runs'].get(RUN_ID, 0)); phase_used = sum(map(float, budget['runs'].values()))
segment_seconds = min(MAX_SEGMENT_MINUTES*60, PER_RUN_GPU_MINUTES*60-run_used, PHASE_C_GPU_MINUTES*60-phase_used)
if segment_seconds <= 0: raise RuntimeError('Pha C GPU budget exhausted for this run or overall.')
# Reserve before launch. If Colab disconnects before cleanup, the full reservation remains charged conservatively.
budget['runs'][RUN_ID] = run_used + segment_seconds; budget_path.write_text(json.dumps(budget, indent=2)+'\n')
cmd = [sys.executable, '-m', 'connect4.selfplay', '--dataset', str(dataset), '--output-root', str(out), '--run-id', RUN_ID, '--seed', str(SEED), '--device', 'cuda', '--iterations', str(MAX_ITERS), '--games-per-iteration', str(GAMES_PER_ITERATION), '--parallel-games', str(PARALLEL_GAMES), '--simulations', str(SIMULATIONS), '--training-steps', str(TRAINING_STEPS), '--batch-size', str(BATCH_SIZE), '--snapshot-interval', str(SNAPSHOT_INTERVAL), '--max-minutes', str(segment_seconds/60)]
if INIT_CHECKPOINT: cmd += ['--init-checkpoint', INIT_CHECKPOINT]
if SMOKE: cmd += ['--smoke']
if (run_dir/'checkpoint_latest.pt').is_file(): cmd += ['--resume']
print('Launching', RUN_ID, 'for at most', segment_seconds/60, 'GPU minutes; budget used before launch:', phase_used/60, 'of', PHASE_C_GPU_MINUTES)
started = time.monotonic()
try:
    subprocess.run(cmd, cwd=repo, check=True)
finally:
    actual = min(segment_seconds, time.monotonic()-started)
    budget = json.loads(budget_path.read_text()); budget['runs'][RUN_ID] = max(0.0, float(budget['runs'][RUN_ID])-segment_seconds+actual); budget_path.write_text(json.dumps(budget, indent=2)+'\n')
    if run_dir.exists():
        shutil.copy2(budget_path, run_dir/'phase_c_gpu_budget.json')
        manifest_path = run_dir/'manifest.json'
        if manifest_path.exists():
            manifest = json.loads(manifest_path.read_text())
            manifest.setdefault('files_sha256', {})['phase_c_gpu_budget.json'] = hashlib.sha256((run_dir/'phase_c_gpu_budget.json').read_bytes()).hexdigest()
            manifest_path.write_text(json.dumps(manifest, indent=2)+'\n')


In [ ]:
# Downloadable package includes snapshots recursively and the budget ledger.
import zipfile
archive = root/'deliveries'/f'{RUN_ID}.zip'; archive.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(run_dir.rglob('*')):
        if p.is_file(): z.write(p, p.relative_to(run_dir.parent).as_posix())
print('artifact:', archive)
